# Chunks zu LLM-Finetuning-Dataset

In diesem Notebook wird aus den vorhandenen Chunks ein einfaches Dataset für das LLM-Finetuning erzeugt. Dafür werden aus jedem Chunk Einträge im Format `instruction` und `response` erstellt.

Das Ziel ist noch kein perfektes Trainingsset mit manuell kuratierten Fragen, sondern ein pragmatischer erster Schritt: Die vorhandenen Texte werden so aufbereitet, dass sie in einem typischen Finetuning-Format vorliegen und als Basis für weitere Verfeinerung dienen.

Die Ausgabe wird als Hugging-Face Dataset im Verzeichnis `dataset_llm` gespeichert.

In [ ]:
from pathlib import Path
from datasets import load_from_disk, Dataset
import shutil

INPUT_DIR = Path("dataset_chunks")
OUTPUT_DIR = Path("dataset_llm")

chunk_ds = load_from_disk(str(INPUT_DIR))
print(chunk_ds)

## Variante 1: Generisches Instruction/Response-Format

Jeder Chunk wird dabei als Wissensantwort gespeichert. Die `instruction` ist generisch gehalten. Das ist einfach, aber für echtes Finetuning meist nur ein Startpunkt.

In [ ]:
rows = []

for row in chunk_ds:
    text = row["chunk"].strip()
    if not text:
        continue

    rows.append({
        "instruction": "Beantworte die Frage anhand des folgenden Fachwissens.",
        "input": "",
        "response": text,
        "path": row.get("path", ""),
        "chunk_id": row.get("chunk_id", -1)
    })

llm_ds = Dataset.from_list(rows)
print(llm_ds)
print(llm_ds[0])

## Optional: Chat-Format ableiten

Viele Chatmodelle verwenden statt `instruction` und `response` eine Nachrichtenstruktur. Diese Zelle erzeugt zusätzlich ein einfaches `messages`-Format.

In [ ]:
chat_rows = []

for row in llm_ds:
    chat_rows.append({
        "messages": [
            {"role": "user", "content": row["instruction"]},
            {"role": "assistant", "content": row["response"]}
        ],
        "path": row["path"],
        "chunk_id": row["chunk_id"]
    })

chat_ds = Dataset.from_list(chat_rows)
print(chat_ds)
print(chat_ds[0])

## Dataset speichern

Standardmässig wird das Instruction/Response-Dataset nach `dataset_llm` gespeichert.

In [ ]:
if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

llm_ds.save_to_disk(str(OUTPUT_DIR))

print(f"Dataset gespeichert unter: {OUTPUT_DIR}")
print(f"Anzahl Einträge: {len(llm_ds)}")

## Optional: Ein paar Beispiele prüfen

In [ ]:
for i in range(min(3, len(llm_ds))):
    print(f"Beispiel {i}")
    print(llm_ds[i])
    print("-" * 80)